In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd
import numpy as np

import tifffile as tf
from tifffile import imread

import skimage
from skimage.morphology import disk
from skimage.morphology import remove_small_holes

import scipy
from scipy.ndimage import binary_dilation
from scipy.ndimage import distance_transform_edt

import matplotlib.pyplot as plt
import matplotlib

from tqdm import tqdm

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
# Import images and csv files

tracks_dir = Path(r'input_path')
tracks_path = os.path.join(tracks_dir,'*.csv')
tracks_files = np.sort(glob.glob(tracks_path))

# Extract the integers before the first underscore in the filenames
loaded_image_ids = {
    os.path.basename(f).split('_')[0]  # Extract the part before the first underscore
    for f in tracks_files
}
cytoplasm_dir = Path(r'input_path')
cytoplasm_path = os.path.join(cytoplasm_dir, '**', '*.tif')  # Use '**' to match subdirectories
cytoplasm_files = np.sort(glob.glob(cytoplasm_path, recursive=True))

# Filter the new images based on matching IDs
cytoplasm_files = np.array([
    f for f in cytoplasm_files
    if os.path.basename(f).split('_')[0] in loaded_image_ids
])

nucleus_dir = Path(r'input_path')
nucleus_path = os.path.join(nucleus_dir,'*.tif')
nucleus_files = np.sort(glob.glob(nucleus_path))

# Filter the new images based on matching IDs
nucleus_files = np.array([
    f for f in nucleus_files
    if os.path.basename(f).split('_')[0] in loaded_image_ids
])

In [ ]:
print(len(tracks_files))
print(len(cytoplasm_files))
print(len(nucleus_files))

In [ ]:
# Read images and csv files into lists

cytoplasm_images = []

for file in tqdm(cytoplasm_files):
    image = imread(file)
    cytoplasm_images.append(image)

nucleus_images = []

for file in tqdm(nucleus_files):
    image = imread(file)
    nucleus_images.append(image)

tracks = []

for file in tqdm(tracks_files):
    tracks_all = pd.read_csv(file)
    tracks.append(tracks_all)

for i in range(len(tracks)):
    tracks[i] = tracks[i].loc[:, ~tracks[i].columns.str.contains('^Unnamed')]

tracks[0].head()

#### Calculate distance to nucleus

In [ ]:
# Compute distance transform

pixel_size = 0.134  # Pixel size in micrometers
default_distance = 150  # Default distance in micrometers for ROIs without a nucleus

for csv, nucleus in zip(tracks, nucleus_images):
    csv['distance_nucleus'] = np.nan  # Create an empty column

    for roi in csv['roi_id'].unique():
        tracks_roi = csv[csv['roi_id'] == roi]
        
        # Check if the nucleus exists for the current ROI
        if roi not in np.unique(nucleus):
            # Assign the default distance for all coordinates in this ROI
            csv.loc[csv['roi_id'] == roi, 'distance_nucleus'] = default_distance
            continue
        
        # Compute the distance transform for the nucleus mask
        mask_distance_transform = scipy.ndimage.distance_transform_edt(~(nucleus == roi))
        
        # Round and clip coordinates
        coords_round = np.round(np.array(tracks_roi[['x', 'y']])).astype(int)
        coords_round[:, 0] = np.clip(coords_round[:, 0], 0, mask_distance_transform.shape[0] - 1)
        coords_round[:, 1] = np.clip(coords_round[:, 1], 0, mask_distance_transform.shape[1] - 1)
        
        # Calculate distances in micrometers
        distances_in_um = mask_distance_transform[coords_round[:, 1], coords_round[:, 0]] * pixel_size
        csv.loc[csv['roi_id'] == roi, 'distance_nucleus'] = distances_in_um


In [ ]:
largest_values = tracks[-1].nlargest(20, 'distance_nucleus')
largest_values

In [ ]:
plt.imshow(mask_distance_transform)

In [ ]:
plt.imshow(nucleus_images[-1]==roi)

#### Merging cytoplasm mask with nucleus

In [ ]:
# Adding nuclei back to cytplasm mask

whole_cells = []

for cyto_mask, nucleus_mask in zip(cytoplasm_images, nucleus_images):

    # Define maximum expansion in pixels
    max_expansion = 70  # Maximum expansion distance in pixels

    # Ensure masks have the same shape
    assert cyto_mask.shape == nucleus_mask.shape, "Masks must have the same dimensions"

    # Get unique nucleus labels
    nucleus_labels = np.unique(nucleus_mask)
    nucleus_labels = nucleus_labels[nucleus_labels > 0]  # Remove background (0)

    # Create a new expanded nucleus mask
    expanded_nucleus_mask = np.copy(nucleus_mask)

    for label_value in nucleus_labels:
        # Extract nucleus region
        nucleus_region = (nucleus_mask == label_value)
        
        # Extract corresponding cytoplasm region
        cyto_region = (cyto_mask == label_value)

        # Initialize dilation step
        dilation_step = 1
        expanded_region = nucleus_region.copy()  # Start with the original nucleus region
        
        while dilation_step <= max_expansion:
            # Apply dilation with a small disk (disk(1) to expand step by step)
            expanded_region = binary_dilation(expanded_region, disk(1))  # Expand by 1 pixel per step
            
            # Check if **all** of the expanded region touches the cytoplasm
            if np.all(expanded_region & cyto_region):  # Stop if ALL of the expanded region touches cytoplasm
                print(f"Expansion stopped at {dilation_step} pixels due to touching cytoplasm.")
                break  # Stop expanding when all of it touches the cytoplasm
            
            dilation_step += 1  # Increment the dilation step
        
        # Assign the expanded region to the output mask
        expanded_nucleus_mask[expanded_region] = label_value  # Update the expanded region in the mask

    # Merge the expanded nuclei with the cytoplasm mask
    whole_cell = np.where(expanded_nucleus_mask > 0, expanded_nucleus_mask, cyto_mask)
    whole_cells.append(whole_cell)

In [ ]:
plt.imshow(whole_cells[0])

In [ ]:
# Removes small holes in the masks

def fill_small_holes_per_label(label_img, area_threshold=64):
    # Create an output array to store the result
    filled = np.zeros_like(label_img)
    
    # Process each unique label (excluding 0, the background)
    for region_label in np.unique(label_img):
        if region_label == 0:
            continue
        mask = label_img == region_label
        filled_mask = remove_small_holes(mask, area_threshold=area_threshold)
        filled[filled_mask] = region_label

    return filled

# Apply to all images in the list
whole_cells_filled = [fill_small_holes_per_label(img, area_threshold=364) for img in whole_cells]


In [ ]:
fig, ax = plt.subplots(1,3, figsize = (15, 8))
ax[0].imshow(whole_cells_filled[3])
ax[1].imshow(whole_cells_filled[4])
ax[2].imshow(whole_cells_filled[5])

#### Calculate distance to edge

In [ ]:
# Calculates distance of each spot to the nearest edge of the mask

def compute_distance_to_edge(label_img, df):
    roi_ids = df['roi_id'].unique()
    distance_maps = {}

    for roi_id in roi_ids:
        mask = label_img == roi_id
        if np.any(mask):
            distance_map = distance_transform_edt(mask)
            distance_maps[roi_id] = distance_map
        else:
            distance_maps[roi_id] = np.zeros_like(label_img, dtype=float)

    distances = []
    for i in range(len(df)):
        row = df.iloc[i]
        x, y, roi_id = int(row['x']), int(row['y']), row['roi_id']
        if roi_id in distance_maps:
            distance_map = distance_maps[roi_id]
            x = np.clip(x, 0, distance_map.shape[1] - 1)
            y = np.clip(y, 0, distance_map.shape[0] - 1)
            distances.append(distance_map[y, x] * 0.134)  # convert pixels to µm
        else:
            distances.append(np.nan)

    df = df.copy()
    df['distance_edge'] = distances
    return df

# Apply with tqdm to track progress

tracks_with_distances = [
    compute_distance_to_edge(label_img, df)
    for label_img, df in tqdm(zip(whole_cells_filled, tracks), total=len(tracks), desc="Computing distances")
]

In [ ]:
tracks_with_distances[0].head()

In [ ]:
# Creates distance maps for all ROIs in an image

def compute_combined_distance_map(label_img, pixel_size_um=0.134):
    distance_map_combined = np.zeros_like(label_img, dtype=float)

    for roi_id in np.unique(label_img):
        if roi_id == 0:
            continue  # skip background
        mask = label_img == roi_id
        if np.any(mask):
            dist_map = distance_transform_edt(mask)
            distance_map_combined[mask] = dist_map[mask] * pixel_size_um  # convert to µm

    return distance_map_combined

In [ ]:
# Display distance transforms

label_img = whole_cells_filled[0]
dist_map_um = compute_combined_distance_map(label_img)

plt.figure(figsize=(8, 6))
plt.imshow(dist_map_um, cmap='viridis')
plt.colorbar(label='Distance to edge (µm)')
plt.title("Distance to Edge Map (µm)")
plt.axis('off')
plt.show()

In [ ]:
# Saving updated csv files

# Base directory and subfolder
base_dir = os.path.dirname(tracks_dir)  # Get the directory of the current file
subfolder = "Dist-nuc_dist-edge"

subfolder_path = os.path.join(base_dir, subfolder)
os.makedirs(subfolder_path, exist_ok=True)  # Create folder if it doesn't exist

for table, file_path in zip(tracks_with_distances, tracks_files):

    # Define the output file path with the correct filename
    output_file = os.path.join(subfolder_path, os.path.basename(file_path))

    # Save the modified CSV
    table.to_csv(output_file, index=False)

print(f"Tables saved in: {subfolder_path}")